# Final Project!

Author: Cameron Mullaney

This project is split into 2 major parts: 
1. Building a Linear Regression Model
2. Using the model to process streaming data

This will all be done using `pyspark` primarily, with pandas used as well. 

In [1]:
import pandas as pd
import numpy as np
import math
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from Proj2Script import *

## Pyspark MLlib-specific tools for model evaluation
from pyspark.ml.regression import LinearRegression
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler, VectorIndexer, OneHotEncoder, PCA
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator 

## Part I - Fitting the Model

Part I is comprised of 3 major tasks:

1. Creating a data transformation pipeline
2. Using that pipeline to fit a LR model
3. Evaluating the LR model.

Starting the spark session

In [2]:
spark = SparkSession.builder.appName('my_app').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 19:38:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Reading the data in as a pandas DF, then converting to spark DF

In [3]:
power = pd.read_csv("power_ml_data.csv")
power = spark.createDataFrame(power) 
power.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

Looks good! Let's get to transforming

### Building the Pipeline

This will be a pretty complicated pipeline, containing many transformations.

For `hour`, I am casting it to a double, and then binarizing it with a cutoff of 6.5

In [4]:
hour_double = SQLTransformer(
    statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_d FROM __THIS__"
)

In [5]:
hour_bin = Binarizer(threshold = 6.5, inputCol = "Hour_d", outputCol = "Hour_b")

For `month`, I am casting it to a double so I can One-hot encode it

In [6]:
month_double = SQLTransformer(
    statement="SELECT *, CAST(Month AS DOUBLE) AS Month_d FROM __THIS__"
)

In [7]:
month_ohe = OneHotEncoder(inputCol = "Month_d", outputCol = "Month_ohe")

Here I'm creating my `label` column

In [8]:
label_cast = SQLTransformer(
    statement = """
                SELECT *, Power_Zone_3 AS label FROM __THIS__
                """
)

Here I'm creating the list of variables I will send to the PCA

In [9]:
PCA_assembler = VectorAssembler(inputCols = ["Temperature", 
                                         "Humidity", 
                                         "Wind_Speed", 
                                         "General_Diffuse_Flows",
                                         "Diffuse_Flows"], 
                            outputCol = "pca_input", 
                            handleInvalid = 'keep')

In [10]:
pca = PCA(k = 2, inputCol = "pca_input", outputCol = "pca_features")

And here I'm running the PCA_assembler to fit the PCA.

In [11]:
temp = PCA_assembler.transform(
    label_cast.transform(power))

pca_fit = pca.fit(temp)        

This is the final assembler, creating the final "features" column for later fitting.

In [12]:
assembler = VectorAssembler(inputCols = ["pca_features", 
                                         "Hour_b", 
                                         "Power_Zone_1", 
                                         "Power_Zone_2",
                                         "Month_ohe"], 
                            outputCol = "features", 
                            handleInvalid = 'keep')

Time to test-run the transformations:

In [13]:
df = hour_double.transform(power)
df = hour_bin.transform(df)
df = month_double.transform(df)
df = month_ohe.fit(df).transform(df)
df = PCA_assembler.transform(df)
df = pca_fit.transform(df)
df = label_cast.transform(df)
temp = assembler.transform(df)

In [14]:
temp.show(5, truncate = False)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+------+------+-------+--------------+------------------------------+----------------------------------------+-----------+-------------------------------------------------------------------------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_d|Hour_b|Month_d|Month_ohe     |pca_input                     |pca_features                            |label      |features                                                                             |
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+------+------+-------+--------------+------------------------------+----------------------------------------+-----------+-------------------------------------------------------------------------------------+
|6.559      |73.8  

This looks good! Time to create our Pipeline

First I'll define what kind of model we want

In [15]:
lr = LinearRegression(featuresCol = "features", labelCol = "label")

In [16]:
pipe_1 = Pipeline(stages = [hour_double, hour_bin, month_double, month_ohe, PCA_assembler, pca, label_cast, assembler, lr])

### Building and fitting the model

Creating our list of potential parameters

In [17]:
LR_paramGrid = ParamGridBuilder()\
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])\
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])\
    .build()


In [18]:
LRCV = CrossValidator(estimator = pipe_1,
                      estimatorParamMaps = LR_paramGrid,
                      evaluator = RegressionEvaluator(metricName = "rmse"),
                      numFolds=5,
                      parallelism = 4,
                      seed = 12)

This is going to take a while - and produce many warnings.

In [19]:
LRCV_model = LRCV.fit(power)

26/04/29 19:38:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/29 19:38:17 WARN Instrumentation: [74f31fe4] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 19:38:17 WARN Instrumentation: [990f455c] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 19:38:17 WARN Instrumentation: [e0df70f6] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 19:38:17 WARN Instrumentation: [43426161] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 19:38:19 WARN Instrumentation: [74f31fe4] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 19:38:19 WARN Instrumentation: [990f455c] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 19:38:19 W

### Reporting model values

Okay! Let's see what our optimal values are

In [30]:
LR_list = []
for i in range(len(LR_paramGrid)):
    LR_list.append([LRCV_model.avgMetrics[i], LR_paramGrid[i].values()])
LR_list.sort(key=lambda x: x[0])
print("CV Error \t\t\t [regParam, ElasticParam]")
LR_list[0]

CV Error 			 [regParam, ElasticParam]


[2147.879982678048, dict_values([0.1, 0.1])]

Training RMSE:

In [31]:
LR_pred = LRCV_model.transform(power)
training_rmse = RegressionEvaluator(metricName = "rmse").evaluate(LR_pred)
print("Training RMSE: \t", training_rmse)

Training RMSE: 	 2147.097320321586


Residuals:

In [32]:
resid_df = LR_pred.withColumn("residual", col("label") - col("prediction"))
resid_df.select("residual", "label", "prediction").show()

+------------------+-----------+------------------+
|          residual|      label|        prediction|
+------------------+-----------+------------------+
|-638.6844049039391|20240.96386| 20879.64826490394|
|1471.1367385456979|20131.08434|18659.947601454303|
|1463.9585040226702|19668.43373| 18204.47522597733|
|1308.8696312355132|18899.27711|17590.407478764486|
|1445.3432101497056|18442.40964|16997.066429850296|
|1612.6517153933491|18130.12048|16517.468764606652|
|1852.0122508872864|17945.06024|16093.047989112712|
|1736.7707088119987|17459.27711|   15722.506401188|
|1754.6696976219446|17025.54217|15270.872472378056|
|1856.0333437770769|16794.21687|14938.183526222923|
| 1985.793232231279|16638.07229|14652.279057768721|
| 1980.376702661104|16395.18072|14414.804017338896|
|2034.8443820286411|16117.59036|14082.745977971359|
|2197.8897606052833| 15822.6506|13624.760839394718|
| 2222.036787445266|15672.28916|13450.252372554734|
| 2294.906820974009|15597.10843|13302.201609025991|
| 2355.57556

## Streaming

Part II is split into two sections - one of which is in "Finalproj_stream.py".

In this notebook, I set up my query and data stream, which will look in the "power_stream" folder for csv's. Then, I'll read in those csv's transform them using `LRCV_model` from part I, and then produce predictions and residuals.

Next, from the same stream, I'll rename "Power_Zone_3" to "label", and join these two tables on this "label" column, and print the result in the console.

In [33]:
powerStream = spark.readStream.csv("power_stream", schema = power.schema, header = True)

Prediction table

In [34]:
stream_predictions = LRCV_model.transform(powerStream) \
    .withColumn("residual", col("label") - col("prediction")) \
    .select("label", "prediction", "residual")

Everything else table (with renamed response column)

In [35]:
stream_label = powerStream.withColumnRenamed("Power_Zone_3", "label")

Now we join them! This one will be used in the final query.

In [26]:
joined_stream = stream_predictions.join(stream_label, on="label")

Great! Now to write the .py file so I can test this!

Now that I've got these ready to go, it's time to start the query, and then run the .py script loop!

In [48]:
query = joined_stream.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

26/04/29 20:15:02 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-3100b1a8-f518-4637-b86a-51d5540bcb96. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/29 20:15:02 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
| 20068.8049| 18642.85301399084| 1425.9518860091594|      21.17|    78.9|     0.281|                0.077|          0.1|  38950.0885| 22887.31809|    9|  23|
|12266.99088| 11199.29905839672|  1067.691821603279|       17.6|    88.7|     4.916|                0.077|        0.133| 27659.34354| 16076.76349|   10|   1|
|36337.40586| 34243.71475620736|   2093.69110379264|      24.12|    83.9|     4.912|                0.051|       

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14173.48315|13547.457579986336|   626.025570013664|      21.37|   59.95|      0.27|                0.055|        0.093| 28678.93805| 17015.80042|    9|   1|
|13538.31325| 14274.03411288627|   -735.72086288627|      13.39|    78.4|     0.079|                0.048|        0.093| 31046.15385| 25359.91736|   11|  23|
|15667.90603|15575.418101377147|  92.48792862285336|      23.24|   42.18|     4.929|                120.0|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15456.30094|19465.928052640273|-4009.6271126402735|      20.72|    76.6|     0.073|                5.088|        3.978| 26070.23307| 16779.72545|    8|   6|
|12734.45783| 14565.07707079348|-1830.6192407934814|      21.94|    57.3|     0.088|                498.5|        109.9| 34012.30769| 24419.00826|   11|  12|
|9882.352941|11541.872093848027|-1659.5191528480282|      16.56|   58.08|     0.087|                209.9|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|11420.89069| 14320.21634887492| -2899.325658874919|       18.6|    81.3|     4.917|                12.33|        10.77| 24097.57377| 16220.43344|    5|   6|
|15986.21538| 17554.61889601258|-1568.4035160125804|      25.49|   57.41|     0.072|                865.0|        214.7| 33224.90066| 21367.98337|    6|  13|
|16345.16129|14766.021751899169| 1579.1395381008315|      17.96|   51.48|     4.921|                0.059|       

...

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
| 13991.8794| 13645.09366608648| 346.7857339135189|      14.38|   63.38|     0.084|                0.059|        0.163| 22899.66102| 13601.21581|    2|   2|
|22882.59109|21600.552449169743|1282.0386408302584|       18.3|    78.7|     0.067|                0.051|        0.159|  37770.4918| 22729.41176|    5|  23|
|15181.21457|15068.377517809646|112.83705219035437|      18.73|    84.3|     4.911|                0.048|        0.14

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16762.38245|18534.896854354556|-1772.5144043545552|      20.87|    75.7|      0.07|                101.6|        40.38| 27553.38513| 19045.40655|    8|   7|
|21969.19305|21130.318959254262|  838.8740907457395|      22.92|    71.0|     0.281|                0.095|        0.067| 42671.15044| 25065.28067|    9|  21|
|21672.72727|20639.473231514094| 1033.2540384859058|      18.55|   69.63|     0.075|                147.7|      

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|24562.49246|25882.946913611093|-1320.4544536110916|      13.15|   55.61|     0.081|                 8.51|         8.46| 43932.20339|  25688.7538|    2|  19|
|13820.46987|13037.671718712934|  782.7981512870665|      23.15|    71.4|     4.915|                237.5|        42.77| 31183.00885| 20945.11435|    9|  17|
|13820.46987|13088.536642084555|   731.933227915446|      20.73|    73.6|     4.917|                0.069|      

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|23328.90282|25436.262312625164|-2107.3594926251644|      28.45|   34.58|     4.905|                810.0|        41.43| 40192.14206| 26477.29673|    8|  12|
|13846.15385|12169.968418710663| 1676.1854312893374|      20.16|    89.0|      4.92|                421.7|        379.2| 26628.19672| 16439.62848|    5|   8|
|    18124.8| 18421.64725244446|-296.84725244446236|      23.34|   38.84|     0.082|                901.0|      

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|11259.54382|10072.352734356753| 1187.1910856432478|      10.14|    84.8|     0.086|                0.073|        0.108| 25332.31939| 20770.78859|   12|   0|
|    18176.0|17877.231662040547| 298.76833795945277|      20.02|   51.92|     0.083|                404.0|        383.4| 32972.74489| 18593.89002|    4|  17|
|27479.27273|26342.115902853722| 1137.1568271462784|      15.78|    83.7|     0.073|                 0.04|      

In [50]:
query.stop()